In [1]:
# =========================================================
# PROJETO: London Crime Analytics
# OBJETIVO:
# Conectar ao BigQuery, extrair dados de crimes em Londres,
# validar qualidade, limpar e exportar datasets analíticos.
# =========================================================

In [2]:
# ==============================================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ==============================================================================
import pandas as pd
import os
import sys
import re
from google.colab import auth
from google.cloud import bigquery

In [3]:
# ==============================================================================
# 2. CONFIGURAÇÃO DO PROJETO E AUTENTICAÇÃO
# ==============================================================================
auth.authenticate_user()

# CRITICAL: Ajuste para o ID do seu projeto criado no GCP
PROJECT_ID = "london-crime-analytics-497414"
SOURCE_TABLE = "bigquery-public-data.london_crime.crime_by_lsoa"

client = bigquery.Client(project=PROJECT_ID)

def run_query(query: str) -> bigquery.job.QueryJob:
    """Executa queries diretamente no BigQuery."""
    return client.query(query)

In [4]:
# ==============================================================================
# 3. AMBIENTE / DEBUG
# ==============================================================================
print('=== Ambiente de Execução ===')
print('Python version:', sys.version.splitlines()[0])
print('Projeto Ativo no GCP:', PROJECT_ID)
print('-' * 40)

=== Ambiente de Execução ===
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Projeto Ativo no GCP: london-crime-analytics-497414
----------------------------------------


In [5]:
# ==============================================================================
# 4. EXTRAÇÃO DO DATASET BRUTO (RAW DATA)
# ==============================================================================
print(">>> [ETAPA 1] Extraindo dados brutos do BigQuery (RAW)...")

sql_raw = f"""
SELECT
    borough,
    major_category,
    minor_category,
    year,
    month,
    SUM(value) AS total_crimes
FROM `{SOURCE_TABLE}`
GROUP BY borough, major_category, minor_category, year, month
ORDER BY year DESC, total_crimes DESC
"""

df_raw = run_query(sql_raw).to_dataframe()

# Garantindo a Rastreabilidade (Salvando a versão RAW original)
os.makedirs("/content/raw", exist_ok=True)
RAW_OUTPUT_PATH = "/content/raw/london_crime_raw.csv"
df_raw.to_csv(RAW_OUTPUT_PATH, index=False)

print(f"Total de registros extraídos (RAW): {len(df_raw)}")
print(f"Dataset RAW salvo com sucesso em: {RAW_OUTPUT_PATH}\n")

>>> [ETAPA 1] Extraindo dados brutos do BigQuery (RAW)...
>>> [ETAPA 1] Extraindo dados brutos do BigQuery (RAW)...
Total de registros extraídos (RAW): 112968
Dataset RAW salvo com sucesso em: /content/raw/london_crime_raw.csv

Total de registros extraídos (RAW): 112968
Dataset RAW salvo com sucesso em: /content/raw/london_crime_raw.csv



In [6]:
# ==============================================================================
# 5. ANÁLISE EXPLORATÓRIA DE DADOS (EDA) - EXCLUSIVO: ENTENDENDO O BUSINESS CASE
# ==============================================================================
print("=" * 60)
print(">>> [ETAPA INTERMEDIÁRIA] ANÁLISE EXPLORATÓRIA VIA SQL")
print("=" * 60)

# Query 1: Preview da Tabela Original
print("\n>>> SQL 1: Preview os dados originais no BigQuery:")
sql_preview = f"SELECT * FROM `{SOURCE_TABLE}` LIMIT 5"
df_res1 = run_query(sql_preview).to_dataframe()
print(df_res1.to_string(index=False))

# Query 2: Contagem de Linhas, Valores Nulos e Consistência de Chaves
print("\n>>> SQL 2: Validação de Nulos e Volume Geral:")
sql_nulls = f"""
SELECT
    COUNT(*) AS total_registros,
    COUNTIF(borough IS NULL) AS nulos_borough,
    COUNTIF(major_category IS NULL) AS nulos_major,
    COUNTIF(minor_category IS NULL) AS nulos_minor,
    COUNTIF(year IS NULL) AS nulos_ano,
    COUNTIF(month IS NULL) AS nulos_mes
FROM `{SOURCE_TABLE}`
"""
df_res2 = run_query(sql_nulls).to_dataframe()
print(df_res2.to_string(index=False))

# Query 3: Resposta de Negócio - Quais macrocategorias de crimes são mais frequentes?
print("\n>>> SQL 3: Distribuição Volumétrica por Macro Categoria (major_category):")
sql_major = f"""
SELECT
    major_category,
    SUM(value) AS total_acumulado,
    ROUND(SUM(value) / SUM(SUM(value)) OVER() * 100, 2) AS percentual
FROM `{SOURCE_TABLE}`
GROUP BY major_category
ORDER BY total_acumulado DESC
"""
df_res3 = run_query(sql_major).to_dataframe()
print(df_res3.to_string(index=False))

# Query 4: Resposta de Negócio - Limites Temporais e Histórico Coberto
print("\n>>> SQL 4: Range Temporal do Dataset:")
sql_time = f"""
SELECT
    MIN(year) AS ano_inicial,
    MAX(year) AS ano_final,
    COUNT(DISTINCT year) AS total_anos_mapeados
FROM `{SOURCE_TABLE}`
"""
df_res4 = run_query(sql_time).to_dataframe()
print(df_res4.to_string(index=False))

# Query 5: Resposta de Negócio - Identificação de Hotspots (Top 5 Bairros com mais ocorrências)
print("\n>>> SQL 5: Top 5 Bairros (Boroughs) Críticos para Alocação de Recursos:")
sql_hotspots = f"""
SELECT
    borough,
    SUM(value) AS volume_crimes
FROM `{SOURCE_TABLE}`
GROUP BY borough
ORDER BY volume_crimes DESC
LIMIT 5
"""
df_res5 = run_query(sql_hotspots).to_dataframe()
print(df_res5.to_string(index=False))

print("\n" + "=" * 60)
print("Análise exploratória via SQL concluída com sucesso!")
print("=" * 60 + "\n")

>>> [ETAPA INTERMEDIÁRIA] ANÁLISE EXPLORATÓRIA VIA SQL

>>> SQL 1: Preview os dados originais no BigQuery:
lsoa_code borough  major_category minor_category  value  year  month
E01000526   Brent Sexual Offences           Rape      0  2008     11
E01000570   Brent Sexual Offences           Rape      0  2008      9
E01000294  Barnet Sexual Offences           Rape      0  2008      3
E01000932  Camden Sexual Offences           Rape      0  2008      3
E01001295  Ealing Sexual Offences           Rape      0  2008     10

>>> SQL 2: Validação de Nulos e Volume Geral:
 total_registros  nulos_borough  nulos_major  nulos_minor  nulos_ano  nulos_mes
        13490604              0            0            0          0          0

>>> SQL 3: Distribuição Volumétrica por Macro Categoria (major_category):
             major_category  total_acumulado  percentual
         Theft and Handling          2661861       41.28
Violence Against the Person          1558081       24.16
                   Burglar

In [7]:
# ==============================================================================
# 6. PIPELINE DE PADRONIZAÇÃO E LIMPEZA (CLEAN DATA)
# ==============================================================================
print(">>> [ETAPA 2] Iniciando a limpeza e padronização dos dados...")

df_clean = df_raw.copy()

# A. Padronização dos nomes das colunas para snake_case
def normalize_column(col: str) -> str:
    col = col.strip().lower()
    col = re.sub(r"[^\w]+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")

df_clean.columns = [normalize_column(c) for c in df_clean.columns]
print("- Colunas padronizadas para snake_case.")

# B. Remoção de linhas e colunas totalmente vazias
cols_before = set(df_clean.columns)
df_clean = df_clean.dropna(axis=1, how='all')
df_clean = df_clean.dropna(axis=0, how='all')
cols_after = set(df_clean.columns)
print(f"- Total de colunas vazias removidas: {len(cols_before - cols_after)}")

# C. Remoção de duplicatas explícitas
linhas_antes = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"- Linhas duplicadas removidas: {linhas_antes - len(df_clean)}")

# D. Ajuste e coerência de tipos de dados
df_clean['year'] = pd.to_numeric(df_clean['year'], errors='coerce').astype('int64')
df_clean['month'] = pd.to_numeric(df_clean['month'], errors='coerce').astype('int64')
df_clean['total_crimes'] = pd.to_numeric(df_clean['total_crimes'], errors='coerce').astype('int64')
print("- Tipos de dados ajustados (`int64`).")

>>> [ETAPA 2] Iniciando a limpeza e padronização dos dados...
- Colunas padronizadas para snake_case.
- Total de colunas vazias removidas: 0
- Linhas duplicadas removidas: 0
- Tipos de dados ajustados (`int64`).


In [ ]:
# ==============================================================================
# 7. SALVANDO DATASET LIMPO (PRONTO PARA O POWER BI)
# ==============================================================================
CLEAN_OUTPUT_PATH = "/content/london_crime_clean.csv"
df_clean.to_csv(CLEAN_OUTPUT_PATH, index=False)

print("\n=== PIPELINE E AUDITORIA CONCLUÍDOS COM SUCESSO ===")
print(f"Arquivo Final Gravado: {CLEAN_OUTPUT_PATH}")
print(f"Linhas finais: {len(df_clean)} | Colunas finais: {df_clean.shape[1]}")


=== PIPELINE E AUDITORIA CONCLUÍDOS COM SUCESSO ===
Arquivo Final Gravado: /content/london_crime_clean.csv
Linhas finais: 112968 | Colunas finais: 6
